In [17]:
import os
import time
import numpy as np
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [18]:
class ACDCDataset(Dataset):
    def __init__(self, root_dir, split="train", transform=None):
        self.root_dir = Path(root_dir)
        self.split = split
        self.transform = transform
        
        self.classes = ["fog", "night", "rain", "snow"]
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}
        
        self.samples = []
        
        for cls in self.classes:
            split_path = self.root_dir / cls / split
            if split_path.exists():
                for img_path in split_path.rglob("*.png"):
                    self.samples.append((img_path, self.class_to_idx[cls]))
        
        print(f"{split} size:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

In [19]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

root = r"C:\Users\Karina\Downloads\ACDC_images\rgb_anon"

train_dataset = ACDCDataset(root, split="train", transform=transform)
val_dataset = ACDCDataset(root, split="val", transform=transform)
test_dataset = ACDCDataset(root, split="test", transform=transform)

train size: 1600
val size: 406
test size: 2000


In [20]:
#гпушки нет, поэтому только на части train обучаемся, иначе долго
indices = list(range(len(train_dataset)))
np.random.shuffle(indices)
subset_size = int(0.2 * len(train_dataset))
train_dataset = Subset(train_dataset, indices[:subset_size])

In [21]:
batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

In [22]:
model = models.resnet18(weights="IMAGENET1K_V1")

model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 128),
    nn.ReLU(),
    nn.Linear(128, 4)
)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [23]:
for epoch in range(5):
    print(f"\n===== EPOCH {epoch+1} =====")
    model.train()

    start = time.time()

    for i, (images, labels) in enumerate(tqdm(train_loader)):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if i % 10 == 0:
            print(f"batch {i} loss {loss.item():.4f}")


===== EPOCH 1 =====


  5%|███████                                                                                                                                       | 1/20 [00:04<01:33,  4.91s/it]

batch 0 loss 1.3651


 55%|█████████████████████████████████████████████████████████████████████████████▌                                                               | 11/20 [00:53<00:43,  4.84s/it]

batch 10 loss 0.8358


...

  5%|███████                                                                                                                                       | 1/20 [00:03<01:15,  3.98s/it]

batch 0 loss 0.0104


 55%|█████████████████████████████████████████████████████████████████████████████▌                                                               | 11/20 [00:47<00:37,  4.20s/it]

batch 10 loss 0.0088


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [01:26<00:00,  4.34s/it]


In [24]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:

        images = images.to(device)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu()

        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

acc = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, average="macro")
rec = recall_score(all_labels, all_preds, average="macro")
f1 = f1_score(all_labels, all_preds, average="macro")

print("VAL RESULTS")
print("accuracy:", acc)
print("precision:", prec)
print("recall:", rec)
print("f1:", f1)

VAL RESULTS
accuracy: 0.9975369458128078
precision: 0.9975247524752475
recall: 0.9975
f1: 0.9974999374984375


In [ ]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

model.eval()
all_preds = []
all_labels = []
skipped_batches = 0

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(test_loader):
        try:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1).cpu()
            
            all_preds.extend(preds.numpy())
            all_labels.extend(labels.numpy())
            
        except Exception as e:
            skipped_batches += 1
            print(f"Пропущен батч {batch_idx} из-за ошибки: {e}")
            continue

acc = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, average="macro", zero_division=0)
rec = recall_score(all_labels, all_preds, average="macro", zero_division=0)
f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

print("TEST RESULTS")
print("accuracy:", acc)
print("precision:", prec)
print("recall:", rec)
print("f1:", f1)